# Spline-Ridge 与 Mean-centered Huber-DML · Lognormal 全量模拟

比较原始 M1 Standard DML、M2 Median-DML、M3 Dukes，以及 S Spline-Ridge-Huber。正式配置为两个 DGP、Gaussian 和 7 档 centered Lognormal、每场景 1000 组，合计 16,000 组数据。Gaussian 用作覆盖下降基准。

使用 **Python (environment2)** 内核，按顺序运行所有单元。Notebook 通过同目录的结果文件夹导入 Windows Worker。Smoke 与正式结果分别持久化。重新运行会从完整断点继续。仅在需要重试数值失败的方法时，将第二单元的 \`RETRY_FAILED\` 设为 \`True\`。

## 1. 固定配置与方法

沿用原五方法 Student-t 实验的 M1、M2、M3，以及最新 Spline-Ridge Pilot 的 S。各方法共享数据及外层五折。S 使用三个内层折、三次 B-spline、5 个分位数结点、常数外推、原线性项和两两交互项。Y、D 分开以平方损失选择 Ridge 参数，各折的变换和标准化只在训练数据拟合。

S 沿用均值位置 Huber profile 求根、OLS-MAD 阈值和位置调整后的 Score-test。真实参数只用于生成、评价和原假设检验。所有正式覆盖率均为 Score-test 不拒绝率。

Lognormal 的 σ 依次为 0.25、0.5、0.75、1.0、1.25、1.5、2.0。以总体均值中心化、总体标准差标准化，保留稳定的误差编号 8–14。非对称误差下，M2 的条件中位数目标与条件均值诊断参照分开解释。

In [ ]:
from pathlib import Path
import sys
import multiprocessing
from IPython.display import Image, Markdown, display

DESKTOP = Path.home() / "Desktop"
EXPERIMENTS = DESKTOP / "毕设DML实验"
RESULTS = EXPERIMENTS / "DML_SplineRidge Lognormal全量实验" / "脚本" / "Spline_Ridge_DML_Lognormal_Results"
SMOKE = RESULTS / "smoke"
sys.path.insert(0, str(RESULTS))

import spline_lognormal_worker as experiment_worker
from spline_lognormal_worker import config_for, smoke_run, full_run
# Worker 的 ROOT 只用于定位历史 Pilot。保留原 Worker 文件和断点签名。
experiment_worker.ROOT = EXPERIMENTS / "DML_SplineRidge预试实验" / "脚本" / "_replay_context"
RETRY_FAILED = False

# 检查 Python、数值包和预先固定的设计。
settings = config_for("full")
print(f"{settings['R']} 次/场景，{len(settings['errors'])} 个误差水平，{settings['workers']} 个 Worker")


## 2. Smoke 自检与历史 Pilot 复放

每个 DGP 运行 Gaussian、LN(0.25)、LN(1)、LN(2) 各一次。自检包含 8 个 Windows 进程、断点读回、CSV、HTML 和图。额外用历史 Pilot 的相同种子复放 8 组 LN(2) 的 S，核对数据、折、Ridge 参数、阈值、点估计与检验。自检数据及历史结果均与正式输出分开。

In [ ]:
if __name__ == "__main__":
    multiprocessing.freeze_support()
    smoke = smoke_run(SMOKE)
    print(f"Smoke 通过：{smoke['data_n']} 组，{smoke['point_n']} 条有效点估计，"
          f"{smoke['test_n']} 条有效检验；历史 S 复放 {len(smoke['pilot_replay'])}/8。")


## 3. 正式 R1000

对 16,000 个任务独立拟合四种方法。每个 Worker 的数值库线程数固定为 1。Notebook 父进程每 32 组保存一个 SQLite 事务，并更新 \`progress.json\` 和 \`run.log\`。原始记录及尝试历史均可追溯，只有父进程写入结果。

In [ ]:
if __name__ == "__main__":
    if not smoke["passed"]:
        raise RuntimeError("先通过 Smoke")
    results = full_run(RESULTS, retry_failed=RETRY_FAILED)
    print(f"正式完成 {results['data_n']:,} 组，点估计有效 {results['point_n']:,}，"
          f"检验有效 {results['test_n']:,}。")


## 4. 核心表与图

这里展示 Gaussian 和最重尾的 LN(2) 场景的核心结果。全部参数、配对比较、失败记录和第一阶段诊断见结果目录的 CSV 和 HTML。

In [ ]:
summary = results["summary"]
main_columns = ["dgp", "error", "method", "coverage_pct", "deterioration_pp",
                "bias", "rmse", "mean_T", "sd_T", "point_n", "test_n"]
display(summary.loc[summary.error_id.isin([0, 14]), main_columns].round(5))
for filename in ("coverage_lognormal.png", "deterioration_lognormal.png",
                 "bias_rmse_lognormal.png"):
    display(Image(filename=str(RESULTS / filename), width=1040))


## Full Simulation Findings

下方结论随本次正式输出自动生成。区分模拟结果、第一阶段误差的机制诊断及尚待验证的理论性质。

In [ ]:
display(Markdown(results["findings"].split("\n\n", 1)[1]))
